In [ ]:
# 23.2 — Book Recommender System / 书籍推荐系统

**Chapter 23 — File 2 of 2 / 第23章 — 第2个文件（共2个）**

## Summary / 总结

Build a book recommender system using SVD-based collaborative filtering. Demonstrates how matrix decomposition can extract latent factors for personalized recommendations.

使用基于SVD的协同过滤构建书籍推荐系统。演示矩阵分解如何提取潜在因子进行个性化推荐。

## Recommender System Concept / 推荐系统概念

**Collaborative Filtering**: If user A and B rate books similarly, recommend books B liked to A
**SVD Approach**: Decompose user-book rating matrix to find latent features and user similarities

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import libraries for recommendation system
# 导入推荐系统库
import tarfile
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

## Step 2 — Load Data from TAR Archive / 从TAR档案加载数据

In [ ]:
# Code pattern for loading book rating data from TAR archive
# TAR档案中加载书籍评分数据的代码模式

example_code = '''
# Extract ratings data
with tarfile.open('ratings.tar.gz', 'r:gz') as tar:
    # List contents
    for member in tar.getmembers():
        print(f"{member.name}")
    
    # Load ratings
    member = tar.getmember('ratings.csv')
    file_obj = tar.extractfile(member)
    ratings = pd.read_csv(file_obj)
    print(f"Ratings shape: {ratings.shape}")
    print(ratings.head())
'''

print("Data Loading Pattern (Book Recommender Dataset):")
print("="*60)
print(example_code)
print()
print("Expected columns: user_id, book_id, rating")

## Step 3 — Create Rating Matrix / 创建评分矩阵

In [ ]:
# Code pattern for creating user-item matrix
# 创建用户项目矩阵的代码模式

example_code = '''
# Create user-book rating matrix
# Rows: users, Columns: books, Values: ratings
rating_matrix = ratings.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating',
    fill_value=0  # Fill missing ratings with 0
)

print(f"Rating matrix shape: {rating_matrix.shape}")
print(f"  Users: {rating_matrix.shape[0]}")
print(f"  Books: {rating_matrix.shape[1]}")
print(f"  Sparsity: {(rating_matrix == 0).sum().sum() / rating_matrix.size * 100:.1f}%")
'''

print("Rating Matrix Creation:")
print("="*60)
print(example_code)

In [ ]:
## Step 4 — Apply SVD for Latent Factors / 应用SVD提取潜在因子

In [ ]:
# SVD for collaborative filtering
# SVD用于协同过滤

example_code = '''
# Apply Truncated SVD to extract latent factors
# Decompose: R ≈ U * Sigma * V^T
# where U: user factors, V: book factors

n_factors = 50  # Number of latent factors
svd = TruncatedSVD(n_components=n_factors)
user_factors = svd.fit_transform(rating_matrix)
book_factors = svd.components_.T

print(f"User factors shape: {user_factors.shape}")
print(f"Book factors shape: {book_factors.shape}")
print(f"\nExplained variance ratio: {svd.explained_variance_ratio_.sum():.3f}")
print(f"This means {svd.explained_variance_ratio_.sum()*100:.1f}% of rating variation is captured")
'''

print("SVD Factorization for Latent Factors:")
print("="*60)
print(example_code)

## Step 5 — Compute User Similarity / 计算用户相似性

In [ ]:
# Compute similarity between users
# 计算用户之间的相似性

example_code = '''
# Compute cosine similarity between users based on latent factors
# Users with similar latent factors have similar tastes
user_similarity = cosine_similarity(user_factors)

print(f"User similarity matrix shape: {user_similarity.shape}")
print(f"\nSimilarity statistics:")
print(f"  Min similarity: {user_similarity[user_similarity < 1].min():.4f}")
print(f"  Max similarity: {user_similarity[user_similarity < 1].max():.4f}")

# Find most similar users
user_id = 0
similar_users = np.argsort(user_similarity[user_id])[-6:-1]  # Top 5 similar users
print(f"\nUsers most similar to user {user_id}:")
for similar_user in similar_users:
    sim_score = user_similarity[user_id, similar_user]
    print(f"  User {similar_user}: similarity = {sim_score:.4f}")
'''

print("Computing User Similarity:")
print("="*60)
print(example_code)

## Step 6 — Generate Recommendations / 生成推荐

In [ ]:
# Generate recommendations
# 生成推荐

example_code = '''
def recommend_books(user_id, n_recommendations=5):
    """
    Recommend books for a user based on similar users' ratings
    基于相似用户的评分为用户推荐书籍
    """
    # Find similar users
    similar_users_idx = np.argsort(user_similarity[user_id])[-11:-1]  # Top 10
    
    # Get books rated by similar users that target user hasn't rated
    # 获取相似用户评分但目标用户未评分的书籍
    user_rated = set(np.nonzero(rating_matrix.iloc[user_id])[0])
    
    # Aggregate recommendations from similar users
    recommendations = {}
    for sim_user in similar_users_idx:
        sim_user_rated = np.nonzero(rating_matrix.iloc[sim_user])[0]
        for book_id in sim_user_rated:
            if book_id not in user_rated:
                if book_id not in recommendations:
                    recommendations[book_id] = []
                sim_score = user_similarity[user_id, sim_user]
                book_rating = rating_matrix.iloc[sim_user, book_id]
                # Weight rating by similarity score
                recommendations[book_id].append(sim_score * book_rating)
    
    # Average scores and rank
    rec_scores = {book: np.mean(scores) for book, scores in recommendations.items()}
    top_books = sorted(rec_scores.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]
    
    return top_books

# Get recommendations
user_id = 0
recommendations = recommend_books(user_id, n_recommendations=5)
print(f"Top 5 book recommendations for user {user_id}:")
for rank, (book_id, score) in enumerate(recommendations, 1):
    print(f"  {rank}. Book {book_id}: score = {score:.4f}")
'''

print("Recommendation Generation:")
print("="*60)
print(example_code)

## Learning Notes / 学习笔记

- **Math Essence**: SVD-based collaborative filtering decomposes the user-item rating matrix into latent factors. Users and items are represented in a low-dimensional space where similarity correlates with rating compatibility. This enables finding "hidden" patterns in user preferences.
  
  **数学本质**：基于SVD的协同过滤将用户项目评分矩阵分解为潜在因子。用户和项目在低维空间中表示，其中相似性与评分兼容性相关。

- **ML Application**: (1) Collaborative filtering works without explicit features, relying only on user-item interactions, (2) SVD reduces dimensionality (50 factors << thousands of books) while preserving rating patterns, (3) Recommendation quality depends on data sparsity - systems with sparse data use more advanced techniques (neural networks, deep learning), (4) This approach powers Netflix, Amazon, and Spotify recommendations at scale.
  
  **ML应用**：(1) 协同过滤无需显式特征，仅依赖用户项目交互，(2) SVD降低维度同时保留评分模式，(3) 推荐质量取决于数据稀疏性，(4) 此方法驱动Netflix、Amazon和Spotify的大规模推荐。

➡️ **Next / 下一步**: `../chapter_24/01_eigenface.ipynb` — Eigenfaces for face recognition

## Complete Code / 完整代码一览

In [ ]:
# --- Import Section / 导入部分 ---
import tarfile
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# --- Load Data from Archive / 从档案加载数据 ---
# with tarfile.open('ratings.tar.gz', 'r:gz') as tar:
#     member = tar.getmember('ratings.csv')
#     file_obj = tar.extractfile(member)
#     ratings = pd.read_csv(file_obj)

# --- Create Rating Matrix / 创建评分矩阵 ---
# rating_matrix = ratings.pivot_table(
#     index='user_id',
#     columns='book_id',
#     values='rating',
#     fill_value=0
# )

# --- SVD and Recommendations / SVD和推荐 ---
# svd = TruncatedSVD(n_components=50)
# user_factors = svd.fit_transform(rating_matrix)
# user_similarity = cosine_similarity(user_factors)

# --- Find Similar Users and Recommend / 查找相似用户和推荐 ---
# user_id = 0
# similar_users = np.argsort(user_similarity[user_id])[-6:-1]
# print(f"Users similar to user {user_id}: {similar_users}")